In [1]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [3]:
data = {
    "text": [
        "The application crashes when I click the login button",
        "Server throws an error when uploading a PDF",
        "The app freezes after submitting the form",
        "Database connection causes the program to crash",
        "Login page gives a 500 error",
        "Add dark mode to the dashboard",
        "Please add Google login support",
        "We need a search bar in the application",
        "Add an option to export reports as PDF",
        "Can you add email notifications?",
        "How do I configure the database connection?",
        "How can I reset my API key?",
        "What are the required environment variables?",
        "How do I deploy this project locally?",
        "Where can I find the authentication documentation?"
    ],

    "label": [
        0,0,0,0,0,
        1,1,1,1,1,
        2,2,2,2,2
    ]
}

df = pd.DataFrame(data)

label_names = {
    0: "Bug",
    1: "Feature Request",
    2: "Question"
}

df["label_name"] = df["label"].map(label_names)

df

,text,label,label_name
0,The application crashes when I click the login...,0,Bug
1,Server throws an error when uploading a PDF,0,Bug
2,The app freezes after submitting the form,0,Bug
3,Database connection causes the program to crash,0,Bug
4,Login page gives a 500 error,0,Bug
5,Add dark mode to the dashboard,1,Feature Request
6,Please add Google login support,1,Feature Request
7,We need a search bar in the application,1,Feature Request
8,Add an option to export reports as PDF,1,Feature Request
9,Can you add email notifications?,1,Feature Request


In [4]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 12
Testing samples: 3


In [5]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=label_names,
    label2id={
        "Bug": 0,
        "Feature Request": 1,
        "Question": 2
    }
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
def compute_metrics(pred):
    predictions = np.argmax(pred.predictions, axis=1)
    labels = pred.label_ids

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [8]:
training_args = TrainingArguments(
    output_dir="./bert_github_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.034433,0.333333,0.111111,0.333333,0.166667
2,No log,0.985563,0.333333,0.111111,0.333333,0.166667
3,No log,0.983691,0.000000,0.000000,0.000000,0.000000


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=9, training_loss=0.9294119940863715, metrics={'train_runtime': 99.8853, 'train_samples_per_second': 0.36, 'train_steps_per_second': 0.09, 'total_flos': 1184010379776.0, 'train_loss': 0.9294119940863715, 'epoch': 3.0})

In [9]:
results = trainer.evaluate()

print("Accuracy :", results["eval_accuracy"])
print("Precision:", results["eval_precision"])
print("Recall   :", results["eval_recall"])
print("F1 Score :", results["eval_f1"])

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,0.983691,3,0.000000,0.000000,0.000000,0.000000


Accuracy : 0.0
Precision: 0.0
Recall   : 0.0
F1 Score : 0.0


In [10]:
def predict_issue(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    return label_names[prediction]

examples = [
    "The website crashes when I upload an image",
    "Please add dark mode",
    "How can I change my password?"
]

for text in examples:
    print(text)
    print("Prediction:", predict_issue(text))
    print()

The website crashes when I upload an image
Prediction: Bug

Please add dark mode
Prediction: Feature Request

How can I change my password?
Prediction: Question

